## **Problem 1: Clean Code Refactoring**

### Objective
Improve readability and maintainability using clean code principles.

### Starter Code
Assume that you have  ```prob1.py``` with the following code

```python
def f(a,b,c):
 x=a+b
 y=x*c
 return y
```

### Tasks
1. rename the function using a meaningful name  
2. rename the variables  
3. improve indentation and readability  

### Questions

1. Why are descriptive names important?
2. How does formatting improve maintainability?
---

## **Problem 2: Modular System Design**

### Objective
Learn how to split a program into modules.

### Starter Code

```python
import statistics

def process(file_path):

    with open(file_path) as file:
        numbers = [int(line.strip()) for line in file]

    mean = statistics.mean(numbers)
    maximum = max(numbers)

    print("Mean:", mean)
    print("Max:", maximum)
```
This program mixes **three responsibilities**:

1. loading data  
2. analysing data  
3. running the program  
> To improve the design, we will split the program into **three modules**: 

Step 1: Create the ```problem2``` Structure

Create the following directory structure.

```
problem2/
│
├── loader.py
├── analysis.py
├── main.py
└── data.txt
```

Step 2: Implement loader.py

This module should contain **only functions related to reading data from files**.

Tasks:

1. Create a function called `load_numbers`
2. The function should read a file
3. It should return a list of integers
4. Do not perform analysis or printing in this module

Step 3: Implement analysis.py

This module should contain **only statistical operations**.

Tasks:

1. Create a function called `compute_mean`
2. Create a function called `compute_max`
3. Both functions should accept a list of numbers
4. Neither function should read files or print output

Step 4: Implement main.py

This file should act as the **entry point of the program**.

Tasks:

1. Import functions from the other modules
2. Call the loader function
3. Call the analysis functions
4. Display the results


Step 5:  Create a Sample Data File

Create a file named **data.txt** inside ```problem2``` directory .

Example content:

```
10
20
15
30
25
```

Step 6:  Run the Program

Execute the program:

```
python main.py
```

Expected output:

```
Mean: 20
Max: 30
```
---

## **Problem 3: Applying SOLID Principles**

### Objective
Apply the single responsibility principle.

### Problematic Code
Assume that you have  ```prob3.py``` with the following code

```python
class Report:

    def generate(self):
        print("Generating report")

    def save(self, path):

        with open(path, "w") as file:
            file.write("Report")
```

### Tasks

1. Identify the design problem.
2. Refactor the code so each class has one responsibility.
---

## **Problem 4: Unit Testing with pytest** 

You are given a simple multiplication function in ```math_utils.py``` . Your task is to write unit tests to verify its behaviour for different inputs, add your testing functions in ```test_math_utils.py```. 

```python
def multiply(a, b):
    return a * b
```
### Tasks

- Write a test for multiplying two positive numbers.

- Write a test for multiplying a number by zero.

- Write a test for multiplying negative numbers.

- Write a test for multiplying floating-point numbers.
---

## **Problem 5: Mocking External Dependencies**

You have a function that fetches a username from an external API in ```api_module.py```. Your task is to write a test without making real HTTP requests. Your task is to write unit tests to verify its behaviour for different inputs, add your testing functions in ```test_api_module.py```. 

> ```api_module.py```
```python
import requests

def get_user_name():
    response = requests.get("https://api.example.com/user")
    return response.json()["name"]
```
### Tasks

- Mock the ```requests.get``` call.
- Simulate a JSON response containing ```{"name": "Ali"}```.
- Verify that ```get_user_name()``` returns ```"Ali"```.
- Test other responses, e.g. missing ```"name"``` key .
---

### Starter Code
> ```test_api_module.py```
```python
from unittest.mock import patch, Mock
import pytest
import requests
from api_module import get_user_name

def test_get_user_name_returns_name():
    with patch("api_module.requests.get") as mock_get:
        mock_response = Mock()
        mock_response.json.return_value = {"name": "Ali"}
        mock_get.return_value = mock_response

        result = get_user_name()

        assert result == "Ali"
        mock_get.assert_called_once_with("https://api.example.com/user")


def test_get_user_name_missing_name_key():
    """TODO for missing name"""

        with pytest.raises(KeyError):
            get_user_name()
```
----

## Take-Home Assignment
Complete the following challenge at home. Remember to split your screen and solve the problems. You may always Google, check the syntax, resolve errors, and brainstorm, but **do not generate code or answers**. See the [submission instructions](https://github.com/SRH-Heidelberg-University-ADSA/Advanced-Python-for-Data-Science/blob/main/SUBMISSION_INSTRUCTIONS.md). 

### Important Announcement
> Starting from this week, you are required to export the chat history of your conversation with any AI tool you choose to use and submit it together with your notebook.

### Mini Challenge: Clean Code, TDD & Mocking
### Scenario

You've inherited a `ReportManager` class from a previous developer:

```python
class ReportManager:
    def __init__(self, api_client):
        self.api_client = api_client

    def run(self):
        raw_data = self.api_client.get_data()

        total = 0
        for record in raw_data:
            total += record["value"]
        average = total / len(raw_data)

        report_text = f"Report\n------\nTotal: {total}\nAverage: {average}\n"

        with open("report.txt", "w") as f:
            f.write(report_text)

        return report_text
```

It works, but it breaks almost every principle from this week:
- It does **too many things** in one method (fetch, process, format, save) → violates **Single Responsibility Principle (SRP)**.
- It's **hard to test** because the API call and the file write are buried inside the same method.
- It has **no tests at all**.

Your job: redesign it, then prove it works with tests including a mocked external dependency.

### Task 1: Refactor for SRP

Split `ReportManager` into **three separate classes**, each with exactly one responsibility:

| Class | Responsibility |
|---|---|
| `DataFetcher` | Wraps a call to `api_client.get_data()` nothing else |
| `ReportProcessor` | Takes raw data, computes `total` and `average`, and formats the report text (pure logic **no network, no file I/O**) |
| `ReportSaver` | Takes report text and writes it to a file |

**Requirements:**
- `DataFetcher` takes `api_client` in its constructor and exposes a `fetch()` method.
- `ReportProcessor` exposes a `process(raw_data)` method returning `(total, average)`, and a
  `format_report(total, average)` method returning a string.
- `ReportSaver` exposes a `save(report_text, filename)` method.
- None of the three classes should do more than one job.

**Expected output:** no runnable output yet, this task is just the class design. Move on to Task 2 to verify it works.
---
### Task 2: TDD the `ReportProcessor` (red → green)

`ReportProcessor` is pure logic, so it's the easiest place to practice TDD.

**Write two tests:**
1. `test_process_computes_total_and_average`, for input
   `[{"value": 10}, {"value": 20}, {"value": 30}]`, assert `total == 60` and `average == 20`.
2. `test_process_single_record` : for input `[{"value": 5}]`, assert `total == 5` and `average == 5`.

**Process:**
- Run your tests **before** `ReportProcessor` is finished → you should see a failure (red stage).
- Finish the implementation until both tests pass (green stage).

**Expected output:**
```
2 passed
```
---
### Task 3: Mock the `DataFetcher` dependency

Test `DataFetcher` **without** making a real API call, using `unittest.mock`.

**Write two tests:**
1. `test_fetch_returns_data_from_api_client`:
   - Create a `Mock()` object standing in for `api_client`.
   - Set `mock_client.get_data.return_value` to a fixed sample list, e.g. `[{"value": 100}]`.
   - Call `DataFetcher(mock_client).fetch()` and assert the result equals your sample data.
   - Assert `get_data()` was called **exactly once** (`assert_called_once()`).
2. `test_fetch_raises_when_api_fails`:
   - Set `mock_client.get_data.side_effect = ConnectionError("API down")`.
   - Assert that calling `fetcher.fetch()` raises `ConnectionError` (`pytest.raises`).

**Expected output:**
```
2 passed
```

### Task 4: Integration test with a mocked API

Combine `DataFetcher` and `ReportProcessor` into one end-to-end test, still using a mocked client (never a real API call).

**Write one test**, `test_full_pipeline_with_mocked_api`:
- Mock client returns `[{"value": 10}, {"value": 20}]`.
- Fetch the data, process it, and format the report.
- Assert the resulting report text contains `"Total: 30"` and `"Average: 15"`.

**Expected output:**
```
1 passed
```
---

**Overall success criteria for the whole challenge:**
- Three classes, each with a single responsibility.
- 5 tests total, all passing.
- At least one test uses `unittest.mock.patch` or `Mock()` so the real API is never called.
- At least one test covers a failure case using `side_effect`.



### Well done, Week 4 Exercises Complete 🎉!

- Don't forget to submit your [exercise notebook](Week4_Exercise.ipynb) on your private GitHub Repository (see [submission instructions](https://github.com/SRH-Heidelberg-University-ADSA/Advanced-Python-for-Data-Science/blob/main/SUBMISSION_INSTRUCTIONS.md)). The deadline is 12.07.2026 @ 23:59. 

- See you next time when we dive into `Web Development & API Design`!